In [1]:
import pandas as pd
import numpy as np
import warnings
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')

# 1. 데이터 로드 (상대 경로 확인)
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

In [2]:
# 2. 결측치 Unknown 범주화
cat_cols = [
    'workclass', 'education', 'marital_status', 'occupation',
    'relationship', 'race', 'sex', 'native_country'
]
for col in cat_cols:
    train_df[col] = train_df[col].fillna("Unknown")
    test_df[col] = test_df[col].fillna("Unknown")

# 3. 수치형 피처 엔지니어링 (v3-a)
for df in [train_df, test_df]:
    df['capital_gain_log'] = np.log1p(df['capital_gain'])
    df['capital_loss_log'] = np.log1p(df['capital_loss'])
    df['has_capital_gain'] = (df['capital_gain'] > 0).astype(int)
    df['has_capital_loss'] = (df['capital_loss'] > 0).astype(int)

# 4. 정답지 분리 및 원-핫 인코딩
y_train = train_df["income"].map({"<=50K": 0, ">50K": 1})
test_id = test_df["id"]

train_features = train_df.drop(columns=["income", "id"])
test_features = test_df.drop(columns=["id"])

full_df = pd.concat([train_features, test_features], axis=0)
full_encoded = pd.get_dummies(full_df, columns=cat_cols)

X_train = full_encoded.iloc[:len(train_df)]
X_test = full_encoded.iloc[len(train_df):]

In [3]:
# 5. 최종 모델 학습 (전체 학습 데이터 100% 사용)
RANDOM_STATE = 42

final_model = XGBClassifier(
    n_estimators=500, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
    random_state=RANDOM_STATE, n_jobs=-1, eval_metric='logloss'
)

final_model.fit(X_train, y_train)

# 6. 최적 Threshold(0.44) 적용 및 prediction.csv 저장
best_threshold = 0.44

y_test_prob = final_model.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_prob >= best_threshold).astype(int)

prediction = pd.DataFrame({
    "id": test_id.values,
    "y_cls": y_test_pred,
    "y_prob": y_test_prob
})

prediction.to_csv("prediction.csv", index=False)

print("최종 산출물 prediction.csv 생성이 완료되었습니다.")
print(f"y_cls 분표: {prediction['y_cls'].value_counts().to_dict()}")

최종 산출물 prediction.csv 생성이 완료되었습니다.
y_cls 분표: {0: 7532, 1: 2237}
